## Adding libraries

In [ ]:
import os
import cv2
import shutil
import numpy as np
import pandas as pd
from glob import glob
import matplotlib.pyplot as plt
import xml.etree.ElementTree as xet
from sklearn.model_selection import train_test_split
import torch

tesseract to read text from image

In [ ]:
!pip3 install pytesseract

Used for Yolo

In [ ]:
!pip install ultralytics

In [ ]:
!pip install -U ipywidgets

DeepSort algorithm tracking

In [ ]:
pip install opencv-python torch torchvision numpy deep_sort_realtime

In [ ]:
import tensorflow as tf
from tensorflow import keras

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from ultralytics import YOLO

library for OCR

In [ ]:
pip install easyocr

In [ ]:
!sudo apt-get install tesseract-ocr

In [ ]:
pip install opencv-python torch torchvision numpy deep_sort_realtime

In [ ]:
!pip install -q datasets jiwer

In [ ]:
!pip install deep-sort-realtime

In [ ]:
pip install gradio

## Creating our own model to detect license plate

### processing data to our first model, detect plate

first step consisted of extracting information from XML file,storing them in dataframe.
Then splitting to training/validation/testing, and storing them in a format required by YOLO to train them

In [ ]:
import re

def the_number_in_the_string(filename):
    """
    Extracts the first sequence of digits from the given filename string and returns it as an integer.
    If no digits are found, returns 0.

    Parameters:
    filename (str): The input string to search for digits.

    Returns:
    int: The first sequence of digits found in the input string, or 0 if no digits are found.
    """
    # Search for the first occurrence of one or more digits in the filename
    match = re.search(r'(\d+)', filename)

    # If a match is found, return the matched number as an integer
    if match:
        return int(match.group(0))
    # If no match is found, return 0
    else:
        return 0

# Example usage
print(the_number_in_the_string("file123.txt"))  # Output: 123
print(the_number_in_the_string("no_numbers_here"))  # Output: 0

In [ ]:


import xml.etree.ElementTree as xet
from glob import glob

# Initialize a dictionary to store labels and image information
labels_dict = dict(
    img_path=[],
    xmin=[],
    xmax=[],
    ymin=[],
    ymax=[],
    img_w=[],
    img_h=[]
)

# Get the list of XML files from the annotations directory
xml_files = glob('annotations/*.xml')

# Process each XML file, sorted by the numerical value in the filename
for filename in sorted(xml_files, key=the_number_in_the_string):
    # Parse the XML file
    info = xet.parse(filename)
    root = info.getroot()

    # Find the 'object' element in the XML and extract bounding box information
    member_object = root.find('object')
    labels_info = member_object.find('bndbox')
    xmin = int(labels_info.find('xmin').text)
    xmax = int(labels_info.find('xmax').text)
    ymin = int(labels_info.find('ymin').text)
    ymax = int(labels_info.find('ymax').text)

    # Get the image filename and construct the full path to the image
    img_name = root.find('filename').text
    img_path = os.path.join('images', img_name)

    # Append the extracted information to the respective lists in the dictionary
    labels_dict['img_path'].append(img_path)
    labels_dict['xmin'].append(xmin)
    labels_dict['xmax'].append(xmax)
    labels_dict['ymin'].append(ymin)
    labels_dict['ymax'].append(ymax)

    # Read the image to get its dimensions
    height, width, _ = cv2.imread(img_path).shape
    labels_dict['img_w'].append(width)
    labels_dict['img_h'].append(height)

# Convert the dictionary to a pandas DataFrame
alldata = pd.DataFrame(labels_dict)

# Display the DataFrame
alldata

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into training and test sets
# Use 10% of the data for the test set
train, test = train_test_split(alldata, test_size=1/10, random_state=42)

# Split the training data further into training and validation sets
# Use 8/9 of the remaining data for the training set, resulting in an 80/10/10 split overall
train, val = train_test_split(train, train_size=8/9, random_state=42)

# Print the number of samples in each set
print(f'''
      len(train) = {len(train)}
      len(val) = {len(val)}
      len(test) = {len(test)}
''')

In [ ]:
train.tail()

now storing in file according to YOLO format.

In [ ]:
def make_split_folder_in_yolo_format(split_name, split_df):
    """
    Creates a folder structure for a dataset split (train/val/test) in YOLO format.

    Parameters:
    split_name (str): The name of the split (e.g., 'train', 'val', 'test').
    split_df (pd.DataFrame): The DataFrame containing the data for the split.

    The function will create 'labels' and 'images' subdirectories under 'datasets/cars_license_plate/{split_name}',
    and save the corresponding labels and images in YOLO format.
    """
    labels_path = os.path.join('datasets', 'cars_license_plate_new', split_name, 'labels')
    images_path = os.path.join('datasets', 'cars_license_plate_new', split_name, 'images')

    # Create directories for labels and images
    os.makedirs(labels_path)
    os.makedirs(images_path)

    # Iterate over each row in the DataFrame
    for _, row in split_df.iterrows():
        img_name, img_extension = os.path.splitext(os.path.basename(row['img_path']))

        # Calculate YOLO format bounding box coordinates
        x_center = (row['xmin'] + row['xmax']) / 2 / row['img_w']
        y_center = (row['ymin'] + row['ymax']) / 2 / row['img_h']
        width = (row['xmax'] - row['xmin']) / row['img_w']
        height = (row['ymax'] - row['ymin']) / row['img_h']

        # Save the label in YOLO format
        label_path = os.path.join(labels_path, f'{img_name}.txt')
        with open(label_path, 'w') as file:
            file.write(f"0 {x_center:.4f} {y_center:.4f} {width:.4f} {height:.4f}\n")

        # Copy the image to the images directory
        shutil.copy(row['img_path'], os.path.join(images_path, img_name + img_extension))

    print(f"Created '{images_path}' and '{labels_path}'")

In [ ]:
# Create YOLO format folders for train, validation, and test splits
make_split_folder_in_yolo_format("train", train)
make_split_folder_in_yolo_format("val", val)
make_split_folder_in_yolo_format("test", test)

adding highlight on training data

In [ ]:

# Directory paths
image_dir = 'datasets/cars_license_plate_new/train/images'
label_dir = 'datasets/cars_license_plate_new/train/labels'

# Get the first image file
image_files = sorted(os.listdir(image_dir))
first_image_file = image_files[0]

# Construct paths for the image and its corresponding label
image_path = os.path.join(image_dir, first_image_file)
label_path = os.path.join(label_dir, os.path.splitext(first_image_file)[0] + '.txt')

# Load the image using OpenCV
image = cv2.imread(image_path)
# Convert the image from BGR (OpenCV default) to RGB (matplotlib default)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Read the label file to get bounding box information
with open(label_path, 'r') as f:
    lines = f.readlines()

# Plot the bounding box on the image
for line in lines:
    # Parse the label file line to extract bounding box information
    class_id, x_center, y_center, width, height = map(float, line.strip().split())
    img_height, img_width, _ = image.shape

    # Convert YOLO format to bounding box format
    x_center *= img_width
    y_center *= img_height
    width *= img_width
    height *= img_height

    # Calculate the top-left and bottom-right coordinates of the bounding box
    x1 = int(x_center - width / 2)
    y1 = int(y_center - height / 2)
    x2 = int(x_center + width / 2)
    y2 = int(y_center + height / 2)

    # Draw the bounding box on the image using a green rectangle
    cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)

# Display the image with bounding box using matplotlib
plt.imshow(image)
plt.axis('off')  # Hide the axis
plt.show()  # Display the image

In [ ]:
# Define the content of the datasets.yaml file
datasets_yaml = '''
path: cars_license_plate_new

train: train/images
val: val/images
test: test/images

# number of classes
nc: 1

# class names
names: ['license_plate']
'''

# Write the content to the datasets.yaml file
with open('datasets.yaml', 'w') as file:
    file.write(datasets_yaml)

### Starting training

we chhose yolo as our detection system.

In [ ]:

model = YOLO('yolov8n.pt')

In [ ]:
model.train(
    data='datasets.yaml',  # Path to the dataset configuration file
    epochs=110,            # Number of training epochs
    batch=16,              # Batch size
    device='cuda',         # Use GPU for training
    imgsz=320,             # Image size (width and height) for training
    cache=True             # Cache images for faster training
)

saving our model

In [ ]:
model.export(format='onnx')

 # loading our best pretrained model

In [ ]:
model_plate_detection = YOLO('best.onnx')

### Testing our model

In [ ]:
import torchvision.transforms as transforms
import pytesseract
from pytesseract import Output
import easyocr


def detect_plate(cf):
    """
    locate car plate, crop it and give to pur model that reads license plate

    Parameters:
    cf (str): frame corresponding to a detected car.

    Returns:
    plate (str,float): return the frame license ex '1234 32', in case of no detection, it returns none.
                      and return s the accuracy of the prediction. 0.0 im case of no results.
    """

    #checking frame
    if cf.shape[0] == 0 or cf.shape[1] == 0:
        return None, 0.0

    #converts the NumPy array into a PIL image.
    cf =Image.fromarray(cf)
    image = cf
    # Resize the image to the expected dimensions (320, 320)
    image = image.resize((320, 320)).convert('RGB')

    # Convert the image to a tensor and normalize
    transform = transforms.Compose([
        transforms.ToTensor(),  # Converts to (3, 320, 320) and scales to [0, 1]
    ])
    image_tensor = transform(image)

    # Add a batch dimension: (1, 3, 320, 320) and acquire the format needed
    image_tensor = image_tensor.unsqueeze(0)
    results = model_plate_detection.predict(image_tensor, device='cpu')



    image_np = np.array(cf)
    # Convert the image from BGR (OpenCV default) to RGB (matplotlib default)
    image_np = cv2.cvtColor(image_np, cv2.COLOR_BGR2RGB)
    image_np = cv2.resize(image_np, (320, 320))

    # Extract the bounding boxes and labels from the results
    for result in results:

        for box in result.boxes:

            # Get the coordinates of the bounding box
            x11, y11, x22, y22 = map(int, box.xyxy[0])
            # Get the confidence score of the prediction
            confidence = box.conf[0]
            x1 = int(x11 )
            y1 = int(y11 )
            x2 = int(x22 )
            y2 = int(y22 )

            # Draw the bounding box on the image
            #cv2.rectangle(image_np, (x1, y1), (x2, y2), (0, 255, 0), 2)
            # Draw the confidence score near the bounding box
            #cv2.putText(image_np, f'{confidence*100:.2f}%', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 0), 2)

            # Crop the license plate region
            plate_frame = image_np[y1:y2, x1:x2]
            #plt.imshow(plate_frame)
            #plt.show()

            #passing the license to our reading model
            text,sc = read_plate(plate_frame)
            print(text)
            print(sc)

            return text , sc
    return None,0.0

In [ ]:
detect_plate()

# Creating our own OCR

## Processing and manipulating dataset

In [ ]:
import os
import time
import numpy as np
import pandas as pd

import cv2
import json
from PIL import Image

output_dir = 'working/output'
os.makedirs(output_dir, exist_ok=True)

In [ ]:
from bs4 import BeautifulSoup

def get_string(file):

    with open(file, 'r') as f:
        data = f.read()

    # Passing the stored data inside the beautifulsoup parser
    bs_data = BeautifulSoup(data, 'xml')

    # Finding all instances of tag
    b_unique = bs_data.find_all('object')
    return ''.join([i.find('name').text for i in b_unique])

def yolo_to_abs(data_dict):#, scale=100.0):
    cord = {}

    original_width = data_dict['original_width']
    original_height = data_dict['original_height']

    pixel_x = int(data_dict['x']/100. * original_width)
    pixel_y = int(data_dict['y']/100. * original_height)
    pixel_width = int(data_dict['width']/100. * original_width)
    pixel_height = int(data_dict['height']/100. * original_height)
#     return[pixel_x, pixel_y, pixel_width, pixel_height]
    cord['x1'] = pixel_x -2#- pixel_width #/2)*original_width
    cord['y1'] = pixel_y -2#- pixel_height#/2)*original_height
    cord['x2'] = pixel_x + pixel_width +5 #/2)*original_width
    cord['y2'] = pixel_y + pixel_height + 5#/2)*original_height
    return cord

In [ ]:
def preprocess(image, width: int, height: int, cval: int = 255, mode="letterbox", return_scale=False,):
    """Obtain a new image, fit to the specified size.
    Args:
        image: The input image
        width: The new width
        height: The new height
        cval: The constant value to use to fill the remaining areas of
            the image
        return_scale: Whether to return the scale used for the image
    Returns:
        The new image
    """
    fitted = None
    x_scale = width / image.shape[1]
    y_scale = height / image.shape[0]
    if x_scale == 1 and y_scale == 1:
        fitted = image
        scale = 1
    elif (x_scale <= y_scale and mode == "letterbox") or (
        x_scale >= y_scale and mode == "crop"
    ):
        scale = width / image.shape[1]
        resize_width = width
        resize_height = (width / image.shape[1]) * image.shape[0]
    else:
        scale = height / image.shape[0]
        resize_height = height
        resize_width = scale * image.shape[1]
    if fitted is None:
        resize_width, resize_height = map(int, [resize_width, resize_height])
        if mode == "letterbox":
            fitted = np.zeros((height, width, 3), dtype="uint8") + cval
            image = cv2.resize(image, dsize=(resize_width, resize_height))
            fitted[: image.shape[0], : image.shape[1]] = image[:height, :width]
        elif mode == "crop":
            image = cv2.resize(image, dsize=(resize_width, resize_height))
            fitted = image[:height, :width]
        else:
            raise NotImplementedError(f"Unsupported mode: {mode}")
    if not return_scale:
        return fitted
    return fitted, scale

In [ ]:
image_dir = 'images'
annotations_dir = 'annotations'

lic_record = []

for im_name in sorted(os.listdir(image_dir)):
    image_path = os.path.join(image_dir, im_name)

    annot_path = os.path.join(annotations_dir, im_name.split('.')[0]+'.xml')
    label_string = get_string(annot_path)

    lic_record.append(dict(text=str(label_string), file_name=image_path))

In [ ]:
df1 = pd.DataFrame(lic_record)
df1.to_csv('lic_labels1.csv')

df1.head()

In [ ]:
license_no_data = pd.read_csv('ocr-licence-plate.csv')

license_no_data['ocr'] = license_no_data['ocr'].apply(lambda x: x.split('-')[-1])

license_no_data

In [ ]:
lic_record = []
path_car_img = "image2"
os.makedirs("working/images/", exist_ok=True)
for row, val in  license_no_data.iterrows():
    x = val['ocr']
    img_dir = os.path.join(path_car_img, x)
    #print("Value of 'x':", x)  # Print the value of x
    #print("Value of 'path_car_img':", path_car_img)  # Print the value of path_car_img
    #print("Resulting 'img_dir':", img_dir)  # Print the constructed img_dir
    image = cv2.imread(img_dir, cv2.IMREAD_ANYCOLOR)
    if image is None:
        print(f"Failed to load image from {img_dir}")
        continue
    try:
        label_s = json.loads(val['transcription'])
    except ValueError:
        label_s = str(val['transcription'])

    print(label_s)
    for num, bbox in enumerate(json.loads(val['bbox'])):
        path_to_save = f"working/images/{val['ocr'].split('.')[0]}_{num}.jpg"

        cart_cord = yolo_to_abs(bbox)
        crop_img = image[cart_cord['y1']:cart_cord['y2'],cart_cord['x1']:cart_cord['x2']]
        crop_img = preprocess(crop_img, width=200, height=100)

#         process_img = cv2.cvtColor(crop_img, cv2.COLOR_RGB2GRAY) #.astype("float32")[..., np.newaxis]
        im_croped = Image.fromarray(crop_img)

        im_croped.save(path_to_save)
        if isinstance(label_s, list):
            lic_record.append( dict(text=str(label_s[num]), file_name=path_to_save))
        else:
            lic_record.append(dict(text=str(label_s), file_name = path_to_save))

In [ ]:
df2 = pd.DataFrame(lic_record)
df2.to_csv('working/lic_labels.csv')
df2.sort_values('file_name', inplace=True)
df2

## Spiltting our dataset

In [ ]:
from sklearn.model_selection import train_test_split

test_df = pd.concat([df1[-21:],df2[-21:]])
df = pd.concat([df1[:-21], df2[:-21]])
# train_df, valid_df = train_test_split(df[:-21], test_size=0.2)
train_df, valid_df = train_test_split(df, test_size=0.2, random_state=0)

# we reset the indices to start from zero
train_df.reset_index(drop=True, inplace=True)
valid_df.reset_index(drop=True, inplace=True)

## Setting up our model to train

This code sets up the TrOCR model by loading the necessary processor for image preprocessing and tokenization, and the VisionEncoderDecoderModel for recognizing and transcribing printed text from images.

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')

# TrOCR is a decoder model and should be used within a VisionEncoderDecoderModel
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-printed')

this code integrates with the TrOCR model for OCR by using a processor to handle image preprocessing and text tokenization. Each sample consists of processed image data and encoded text labels, prepared for training with a model that uses sequence-to-sequence learning.

In [ ]:
import torch
from torch.utils.data import Dataset

class LPDataset(Dataset):
    def __init__(self, root_dir, df, processor, max_target_length=128):
        self.root_dir = root_dir
        self.df = df
        self.processor = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # get file name + text
        file_name = self.df['file_name'][idx]
        text = self.df['text'][idx]

        # prepare image (i.e. resize + normalize)
        image = Image.open(self.root_dir + file_name).convert("RGB")
        pixel_values = self.processor(image, return_tensors="pt").pixel_values

        # add labels (input_ids) by encoding the text
        labels = self.processor.tokenizer(
            text, padding="max_length", max_length=self.max_target_length).input_ids

        # important: make sure that PAD tokens are ignored by the loss function
        labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]

        encoding = {"pixel_values": pixel_values.squeeze(), "labels": torch.tensor(labels)}
        return encoding

    def __iter__(self):
        for i in range(self.__len__()):
            yield self.__getitem__(i)

In [ ]:
root_dir = ''

train_dataset = LPDataset(root_dir, df=train_df, processor=processor)

test_dataset = LPDataset(root_dir, df=test_df, processor=processor)

eval_dataset = LPDataset(root_dir, df=valid_df, processor=processor)

It first loads the TrOCR processor for handling the image preprocessing and tokenization. The image is processed into pixel values, which are then passed to the TrOCR model to generate predictions. Finally, the generated text IDs are decoded back into a human-readable string, which represents the recognized text from the image.

In [ ]:
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')
pixel_values = processor(images=image, return_tensors="pt").pixel_values

generated_ids = model.generate(pixel_values)
generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [ ]:
# set special tokens used for creating the decoder_input_ids from the labels
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
# make sure vocab size is set correctly
model.config.vocab_size = model.config.decoder.vocab_size

# set beam search parameters
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.max_length = 64
model.config.early_stopping = True
model.config.no_repeat_ngram_size = 3
model.config.length_penalty = 2.0
model.config.num_beams = 4

choosing metric, CER since we are going for an OCR.

In [ ]:
from datasets import load_metric

cer_metric = load_metric("cer")

In [ ]:
def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)

    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"cer": cer}

parameters for training

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, default_data_collator

training_args = Seq2SeqTrainingArguments(
    predict_with_generate=True,      # Enables text generation during evaluation
    evaluation_strategy="steps",    # Evaluates at specific steps
    per_device_train_batch_size=8,  # Training batch size per device
    per_device_eval_batch_size=8,   # Evaluation batch size per device
    overwrite_output_dir=True,      # Overwrites existing output directory
    output_dir=output_dir,         # Directory to save model outputs
    logging_steps=2,              # Logs every 2 steps
    save_steps=500,             # Saves model checkpoint every 500 steps
    eval_steps=200,             # Evaluates every 200 steps
    num_train_epochs=4,        # Number of training epochs
)


# instantiate trainer
trainer = Seq2SeqTrainer(
    model=model,
    tokenizer=processor.image_processor,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=default_data_collator,  # Collates data into batches
    )

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained("vit-ocr")

## Load a pretrained model





In [ ]:
import torch
from PIL import Image
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from difflib import SequenceMatcher

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
model_read_plate = VisionEncoderDecoderModel.from_pretrained('/content/drive/MyDrive/MyFolder')
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")

function to read plate

In [ ]:
import torch.nn.functional as F
def read_plate(plate_frame):

    """
    read plate, and return the result to detect_plate function.

    Parameters:
    cf (str): frame corresponding to a detected license plate.

    Returns:
    plate (str,float): return the frame license ex '1234 32', in case of incapability of reading, it returns none.
                      and return s the accuracy of the prediction. 0.0 im case of no results
    """

    image =Image.fromarray(plate_frame)

    # convert("RGB")
    image = image.convert("RGB")

    pixel_values = processor(image, return_tensors="pt").pixel_values
    # if torch.cuda.is_available():
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    model_read_plate.to(device)
    pixel_values = pixel_values.to(device)

    generated_ids = model_read_plate.generate(pixel_values, output_scores=True, return_dict_in_generate=True)
    output = generated_ids.sequences
    generated_text = processor.batch_decode(output, skip_special_tokens=True)[0]
    if generated_text== "":
      average_confidence =0.0
      return 'None',average_confidence

    # get score
    logits = generated_ids.scores
    #Calculate prediction scores (confidence)
    # Taking the softmax to convert logits to probabilities for each token
    logits = torch.stack(logits, dim=0)
    probabilities = F.softmax(logits, dim=-1)
    predicted_probs = probabilities.max(dim=-1).values  # Highest probability for each token

    # Aggregate the probabilities to represent the whole sequence (mean as a simple method)
    average_confidence = predicted_probs.mean().item()
    print('byeeeee')

    return generated_text,average_confidence

In [ ]:

read_plate(c2)
#tensor([[    0,     0, 10842,   510,    12,  3196,   306,     2]])

# detection and tracking






deep sort tracking algorithm

In [ ]:

import torchvision.transforms as transforms
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from google.colab.patches import cv2_imshow

In [ ]:
from torchvision.ops import nms
from deep_sort_realtime.deepsort_tracker import DeepSort

nms_threshold = 0.5   # for overlapping
CONFIDENCE_THRESHOLD = 0.7  # Confidence threshold for detection

# Load the pre-trained model (Faster R-CNN in this case)
model2 = fasterrcnn_resnet50_fpn(pretrained=True)
model2.eval()


# Transform for input images
transform = transforms.Compose([
    transforms.ToTensor(),
])


# COCO class IDs for car, truck, and motorcycle
TARGET_CLASSES = [3, 4, 8]  # Car, Motorcycle, Truck

COCO_CLASSES = [
    '_background_', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign',
    'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag',
    'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite',
    'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana',
    'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
    'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table',
    'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone',
    'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock',
    'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]

# Function to preprocess the frame
def preprocess_frame(frame):
    return transform(frame).unsqueeze(0)

#set color depending of label
def get_label_color(label):
    if label == "truck":
        return (0, 255, 0)  # Green
    elif label == "car":
        return (0, 0, 255)  # Red
    else:
        return (255, 0, 0)  # Blue

#crop car frame
def crop_car(image,x1,y1,x2,y2):

    min_x = x1
    min_y = y1
    max_x = x2
    max_y = y2

    # Crop the face out of the car
    car = image[int(min_y) : int(max_y), int(min_x) : int(max_x)]

    return car


# Function to draw bounding boxes on the frame
def draw_boxes(frame, outputs, confidence_threshold,tracker,car_dic,veh_set):
    """
    Draw bounding boxes on the frame based on the detection results. it used all of our models described above.

    Parameters:
    frame (str): frame corresponding to a video.
    outputs (str): detection results.
    confidence_threshold (float): confidence threshold for detection.
    tracker (class tracker): tracker object fr\or deepsrt algorithm.
    car_dic (dictionnary): dictionnary containing the vehicules track id and their info.
    veh_set (set): set containing the unique vehicules track id.

    Returns:
    plate (dictionnary): return the car_dic containing the vehicules and their info.
                         draw boxes on the frame for each detections.
    """
    boxes  = []
    scores = []
    labels = []
    for box, label, score in zip(outputs['boxes'], outputs['labels'], outputs['scores']):
        if score > confidence_threshold and label in [3, 4, 8]:  # Assuming vehicle classes: car, truck, motorcycle
            boxes.append(box.cpu())
            scores.append(score.cpu().item())
            labels.append(label.cpu())

    #stored detections results and their info.
    boxes_tensor = torch.stack(boxes)
    scores_tensor = torch.tensor(scores)
    labels_tensor = torch.tensor(labels)

    # Apply NMS, reduce overalpping.
    keep_indices = nms(boxes_tensor, scores_tensor, nms_threshold)

    # Filter the boxes, scores, and labels using NMS results
    boxes = boxes_tensor[keep_indices].numpy()
    scores = scores_tensor[keep_indices].numpy()
    labels = [COCO_CLASSES[l.cpu().item()] for l in labels_tensor]

    # Convert boxes to the format expected by DeepSORT
    detections = [([box[0], box[1], box[2], box[3]], score,label) for box, score,label in zip(boxes, scores,labels)]

    # Update tracker with new detections
    tracks = tracker.update_tracks(detections, frame=frame,others=detections)

    # Draw tracking results on the frame
    #tracked_veh_numb = len(tracks)
    for track in tracks:

        track_id = track.track_id
        x1, y1, x2, y2 = track.to_tlbr(orig=True)  # Get bounding box coordinates in (x1, y1, x2, y2) format
        w = x2 - x1  # Calculate width
        h = y2 - y1  # Calculate height
        car_frame = crop_car(frame,x1,y1,w,h)
        #plt.imshow(cv2.cvtColor(car_frame, cv2.COLOR_BGR2RGB))  # Convert from BGR to RGB
        #plt.show()
        info = track.get_det_supplementary()
        #print(info)
        #print(x1)
        #print(y1)
        #print(x2)
        #print(y2)
        #print(w)
        #print(h)

        current_plate,plate_score = detect_plate(car_frame)

        if info is None:
              continue


        # algorithm to keep the best result for each vehicle for both labels and plate number, more details in read me file.
        current_label = info[2]
        scores_veh = info[1]

        if track_id in veh_set:
              inf = car_dic[track_id]

              if inf[0]< scores_veh:
                car_dic[track_id][0] = scores_veh
                car_dic[track_id][1] = current_label

              if inf[2] < plate_score:
                car_dic[track_id][2] = plate_score
                car_dic[track_id][3] = current_plate

        else:
            car_dic[track_id] = [scores_veh,current_label,plate_score,current_plate]
            veh_set.add(track_id)


        # Get color based on label
        print(car_dic[track_id][0])
        print(car_dic[track_id][1])
        print(car_dic[track_id][2])
        print(car_dic[track_id][3])
        print(veh_set)
        col = get_label_color(current_label)
        # Draw bounding box and ID
        current_label = car_dic[track_id][1]
        current_plate = car_dic[track_id][3]

        cv2.rectangle(frame, (int(x1), int(y1)), (int(w), int(h)), col, 2)
        cv2.putText(frame, f'{current_label}: {track_id} Plate : {current_plate}', (int(x1) + 10, int(y1) - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 2)

    return car_dic


In [ ]:
def Main(video):
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    cap = cv2.VideoCapture(video)


    tracker = DeepSort(
    max_cosine_distance=1.0,             # Maximum cosine distance for association
    nms_max_overlap=1.0,                 # Maximum overlap for Non-Maximum Suppression
    max_iou_distance=0.7,                # Maximum IoU distance for association
    max_age=35,                          # Maximum age before a track is considered lost
    n_init=5,                            # Minimum number of frames to confirm a track
    nn_budget=300                        # Maximum size of the appearance gallery
    )
    car_dic = {}
    veh_set = set()


    # Get the video properties
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    summary_car ={}

    # Create the video writer object for an MP4 file
    out = cv2.VideoWriter('output_video.mp4', fourcc, fps,(w , h))

    while True:
       ret, frame = cap.read()
       if not ret:
          break

       # Preprocess the frame
       input_tensor = preprocess_frame(frame)

       # Run the frame through the model
       with torch.no_grad():
          detections = model2(input_tensor)[0]  # Get the first item in the batch
       # Draw the bounding boxes on the frame
       summary_car = draw_boxes(frame, detections, CONFIDENCE_THRESHOLD,tracker,car_dic,veh_set)

       # Show the frame
       cv2_imshow( frame)

       # Write the frame to the output video
       out.write(frame)

       # Exit loop if 'q' is pressed
       if cv2.waitKey(1) & 0xFF == ord('q'):
          break

       # Release video capture, writer, and close windows
    cap.release()
    out.release()
    cv2.destroyAllWindows()
    result_string = ""

    # Iterate over the dictionary to display all our info
    print(summary_car)
    for track_id, values in summary_car.items():
       entry_string = f"Track ID: {track_id}, Score Vehicle: {values[0]}, Current Label: {values[1]}, Plate Score: {values[2]}, Current Plate: {values[3]}"
       result_string += entry_string + "\n"
    print(result_string)
    return 'output_video.mp4',result_string

In [ ]:
#video,report = Main("/content/WhatsApp Video 2024-08-28 at 12.01.20 PM.mp4")

INTERFACE

In [ ]:
import gradio as gr

In [ ]:


interface = gr.Interface(
    fn=Main,
    inputs=gr.Video(),
    outputs=[gr.Video(),"textbox"],
    title="Vehicle Detection",
    cache_examples=False,
    description="Upload a video to detect vehicles, track them, and read their plate numbers."
)

interface.launch(debug=True)